# Convert mcPHASES CSVs to parquet (one-off setup)

Run this notebook **once**, after placing the mcPHASES dataset in your Google Drive (Colab) or local `dataset/` folder. Subsequent loads in any other notebook (`utils.dataset.load_mcphases()`) will use the parquet versions automatically.

**What this notebook does**

1. Mounts Google Drive (Colab only).
2. Locates your raw mcPHASES CSVs.
3. Converts the nine large files (heart_rate, calories, wrist_temperature, distance, steps, estimated_oxygen_variation, sleep, glucose, heart_rate_variability_details) to compressed parquet — about **3.4 GB CSV → 340 MB parquet**, ~30 seconds on a laptop.
4. Verifies one converted file by reading a single participant's data.

Smaller files (hormones, subject info, sleep score, etc.) stay as CSV — no benefit from converting them.

**This is idempotent.** Re-running skips files whose parquet copy already exists; pass `--force` if you need to rebuild.

## 1. Setup — repo path, Drive mount, imports

In [ ]:
import sys, os, subprocess
from pathlib import Path

# Walk up from cwd to find the repo root (works from any directory)
_p = Path().resolve()
while not (_p / 'utils' / 'dataset.py').is_file() and _p != _p.parent:
    _p = _p.parent
REPO_ROOT = _p
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    on_colab = True
except ImportError:
    on_colab = False
    print('Local Jupyter — Drive mount skipped.')

print(f'Repo root: {REPO_ROOT}')

## 2. Locate the dataset

The script searches for `dataset/` next to the repo, but on Colab the data usually lives on Drive. **Edit `DATASET_DIR` below** to point at your folder. The cell will check that the expected files are there before doing anything destructive.

In [ ]:
# === EDIT THIS LINE if your dataset is somewhere else ===
DATASET_DIR = Path('dataset')  # local layout default

# Colab users typically have something like:
# DATASET_DIR = Path('/content/drive/MyDrive/mcphases-1.0.0')

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(
        f'{DATASET_DIR} not found. Edit DATASET_DIR above to point at your raw mcPHASES folder. '
        f'On Colab the path usually starts with /content/drive/MyDrive/...'
    )

csvs = sorted(DATASET_DIR.glob('*.csv'))
if not csvs:
    raise RuntimeError(f'No CSV files in {DATASET_DIR}. Wrong folder?')

print(f'Found {len(csvs)} CSV files in {DATASET_DIR}')
total_mb = sum(p.stat().st_size for p in csvs) / 1024**2
print(f'Total size: {total_mb:,.0f} MB')
for p in csvs:
    mb = p.stat().st_size / 1024**2
    flag = '  -> will convert' if mb >= 10 else ''
    print(f'  {p.name:<45s} {mb:>8.1f} MB{flag}')

## 3. Run the conversion

Output goes to `dataset_parquet/` next to the source CSVs (e.g., `<your-folder>_parquet/` on Drive). Progress prints as each chunk lands. Expect ~30 seconds locally; on Colab the heart_rate file may take 1–2 minutes depending on Drive throughput.

In [ ]:
PARQUET_DIR = DATASET_DIR.parent / f'{DATASET_DIR.name}_parquet'

result = subprocess.run(
    [sys.executable, 'scripts/convert_raw_to_parquet.py',
     '--src', str(DATASET_DIR),
     '--dst', str(PARQUET_DIR)],
    text=True,
)
if result.returncode != 0:
    raise SystemExit(f'Conversion failed with exit {result.returncode}')

print(f'\nParquet output directory: {PARQUET_DIR}')

## 4. Verify outputs

List the produced files and check that loading one of them works.

In [ ]:
import pandas as pd

outputs = sorted(PARQUET_DIR.glob('*.parquet'))
print(f'Produced {len(outputs)} parquet files:')
for p in outputs:
    mb = p.stat().st_size / 1024**2
    print(f'  {p.name:<45s} {mb:>7.1f} MB')

# Smoke-test: load one participant's heart_rate from the converted file
hr_path = PARQUET_DIR / 'heart_rate.parquet'
if hr_path.is_file():
    sample = pd.read_parquet(hr_path,
                              columns=['id', 'day_in_study', 'bpm'],
                              filters=[('id', '==', 1)])
    print(f'\nLoaded {len(sample):,} heart_rate rows for participant 1.')
    print(sample.head())

## 5. Next

From this point onwards, every notebook (`01_wp2_preprocessing`, `04_wp6_conformal`, etc.) calls `utils.dataset.load_mcphases()` and gets fast, Colab-friendly access to the same data. The conversion never needs to run again unless the raw CSVs change.

If you set a non-default `DATASET_DIR` above, pass it through to the loader once:

```python
from utils.dataset import load_mcphases
data = load_mcphases(data_dir='/content/drive/MyDrive/mcphases-1.0.0')
```

If the path matches one of the auto-detected locations (typed in `utils/dataset.py`), no argument is needed.

**Walkthrough of the loader is in `notebooks/06_dataset_loading.ipynb`.**